In [ ]:
!pip install -q uv
!uv pip install --system moshi speechbrain sentencepiece pymcd resemblyzer torchaudio==2.10.0

In [1]:
import os
import torch
import torch.nn as nn
import torchaudio
import sentencepiece
import librosa
import numpy as np
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from torch.amp import autocast

try:
    from resemblyzer import VoiceEncoder, preprocess_wav
except ImportError:
    print("⚠️ Missing resemblyzer! Run: !pip install resemblyzer")

# ==========================================
# 0. THE DYNAMIC GRAPH KILLER
# ==========================================
import moshi.utils.compile

print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'): return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

# ==========================================
# 1. SETUP & EXACT TRUE SOUNDSTORM ARCHITECTURE
# ==========================================
DEVICE_HOME = "cuda:0" 
DEVICE_WORK = "cuda:1" 

# 🚀 Points to the new True SoundStorm weights
BRIDGE_PATH = "/kaggle/input/datasets/ahmedsadman099876/model9867/true_soundstorm_best.pt" 
TARGET_PT_PATH = "/kaggle/input/datasets/ahmedsadman099876/data899/0012_001164.pt"
USER_PROMPT_WAV = "/kaggle/input/datasets/ahmedsadman099876/recording/Recording.wav"
REFERENCE_WAV = "/kaggle/input/datasets/ahmedsadman099876/data458/0012_001164.wav"
OUTPUT_FILENAME = "/kaggle/working/final_bridged_eval_SoundStorm.wav"

class TrueSoundStormBridge(nn.Module):
    def __init__(self, vocab_size=2048, embed_dim=1024, id_dim=192, max_len=4096):
        super().__init__()
        self.cb0_embedding = nn.Embedding(vocab_size, embed_dim)
        self.id_projection = nn.Linear(id_dim, embed_dim)
        self.positional_encoding = nn.Embedding(max_len, embed_dim)
        
        self.cnn_prenet = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU()
        )
        
        self.temporal_transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=8, batch_first=True, dropout=0.1, norm_first=True 
            ), num_layers=2
        )
        
        self.rvq_feedback_embs = nn.ModuleList([nn.Embedding(vocab_size, embed_dim) for _ in range(6)])
        self.acoustic_heads = nn.ModuleList([nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, vocab_size)) for _ in range(7)])

    def apply_ppmq_adain(self, cb0_emb, id_vec, residual_factor, identity_factor):
        """Prosody-Preserving Mixed Quantization (PPMQ) - Tunable Mix"""
        id_raw = self.id_projection(id_vec).unsqueeze(1)
        id_min, id_max = id_raw.min(dim=-1, keepdim=True)[0], id_raw.max(dim=-1, keepdim=True)[0]
        scale = (id_max - id_min) / 15
        id_quant = torch.round((id_raw - id_min) / (scale + 1e-8)) * scale + id_min
        
        
        cb0_mean = cb0_emb.mean(dim=-1, keepdim=True)
        cb0_var = ((cb0_emb - cb0_mean) ** 2).mean(dim=-1, keepdim=True)
        cb0_std = torch.sqrt(cb0_var + 1e-5)
        
        id_mean = id_quant.mean(dim=-1, keepdim=True)
        id_var = ((id_quant - id_mean) ** 2).mean(dim=-1, keepdim=True)
        id_std = torch.sqrt(id_var + 1e-5)
        
        identity_payload = cb0_std * ((id_quant - id_mean) / id_std) + cb0_mean
        return (cb0_emb * residual_factor) + (identity_payload * identity_factor)

    def forward_infer(self, cb0_tokens, id_vec):
        """TRUE SOUNDSTORM INFERENCE: Fast Parallel Cascading."""
        if cb0_tokens.dim() == 3: cb0_tokens = cb0_tokens.view(cb0_tokens.shape[0], -1)
        seq_len = cb0_tokens.size(1)
        device = cb0_tokens.device
        
        positions = torch.arange(0, seq_len, device=device).unsqueeze(0)
        cb0_emb = self.cb0_embedding(cb0_tokens) + self.positional_encoding(positions)
        
        # 🛠️ THE DIALS: 80% Moshi Structure, 120% Target Identity Overdrive
        latent_stream = self.apply_ppmq_adain(cb0_emb, id_vec, residual_factor=0.80, identity_factor=1.20)
        
        logits_list = []
        for i in range(7):
            temp_stream = latent_stream.transpose(1, 2)
            temp_stream = self.cnn_prenet(temp_stream)
            temp_stream = temp_stream.transpose(1, 2)
            temporal_features = self.temporal_transformer(temp_stream)
            
            layer_logits = self.acoustic_heads[i](temporal_features)
            logits_list.append(layer_logits)
            
            if i < 6:
                predicted_tokens = layer_logits.argmax(dim=-1)
                latent_stream = latent_stream + self.rvq_feedback_embs[i](predicted_tokens)
                
        return torch.stack(logits_list, dim=2)

# ==========================================
# 2. LOAD MODELS & CONSTRUCT SANDWICH
# ==========================================
from moshi.models import loaders, LMGen

repo_id = "kyutai/moshiko-pytorch-bf16"
print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"), device="cpu")
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)

print("Downloading Moshi weights (~15.4 GB)...")
moshi = loaders.get_moshi_lm(hf_hub_download(repo_id, "model.safetensors"), device="cpu")
text_tokenizer = sentencepiece.SentencePieceProcessor(hf_hub_download(repo_id, "tokenizer_spm_32k_3.model"))

print("\n🛠️ Sharding Model across GPUs...")
def shard_hook(module, args):
    try: target_dev = next(module.parameters()).device
    except StopIteration:
        try: target_dev = next(module.buffers()).device
        except StopIteration: return args
    if isinstance(args, tuple): return tuple(a.to(target_dev) if isinstance(a, torch.Tensor) else a for a in args)
    return args.to(target_dev) if isinstance(args, torch.Tensor) else args

mid = len(moshi.transformer.layers) // 2
for i in range(mid, len(moshi.transformer.layers)):
    moshi.transformer.layers[i].to(DEVICE_WORK)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)

moshi.emb.to(DEVICE_HOME)
moshi.text_emb.to(DEVICE_HOME)
for i in range(mid):
    moshi.transformer.layers[i].to(DEVICE_HOME)
    moshi.transformer.layers[i].register_forward_pre_hook(shard_hook)
if hasattr(moshi.transformer, "norm"):
    moshi.transformer.norm.to(DEVICE_HOME)
    moshi.transformer.norm.register_forward_pre_hook(shard_hook)
for name, module in moshi.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

for name, buf in moshi.named_buffers(recurse=True): buf.data = buf.data.to(DEVICE_HOME)
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi, attr):
        t = getattr(moshi, attr)
        if isinstance(t, torch.Tensor): setattr(moshi, attr, t.to(DEVICE_HOME))

lm_gen = LMGen(moshi, temp=0.8, temp_text=0.7)

# ==========================================
# 3. PHASE 1: NATIVE GENERATION (SMART TRIMMER)
# ==========================================
try:
    wav, sr = torchaudio.load(USER_PROMPT_WAV)
    if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
    if sr != 24000: wav = torchaudio.functional.resample(wav, sr, 24000)
    wav = wav.unsqueeze(0).to(DEVICE_HOME)
    silence_padding = torch.zeros(1, 1, int(24000 * 0.5)).to(DEVICE_HOME)
    wav = torch.cat([wav, silence_padding], dim=-1)
except FileNotFoundError:
    wav = torch.zeros(1, 1, 24000 * 3).to(DEVICE_HOME)


print("\n⚡ Phase 1: Generating Base Tokens Natively...")
frame_size = mimi.frame_size
all_codes = []
with torch.no_grad(), mimi.streaming(batch_size=1):
    for offset in range(0, wav.shape[-1], frame_size):
        frame = wav[:, :, offset : offset + frame_size]
        if frame.shape[-1] < frame_size: frame = torch.nn.functional.pad(frame, (0, frame_size - frame.shape[-1]))
        all_codes.append(mimi.encode(frame))

collected_moshi_codes = []
generated_text_pieces = []

with torch.no_grad(), lm_gen.streaming(1), mimi.streaming(1):
    def process_step(tokens_out):
        text_token = tokens_out[0, 0].item()
        if text_token not in (0, 3): 
            generated_text_pieces.append(text_tokenizer.id_to_piece(text_token).replace(' ', ' '))
        collected_moshi_codes.append(tokens_out[:, 1:])

    for code in all_codes:
        tokens_out = lm_gen.step(code)
        if tokens_out is not None: process_step(tokens_out)

    print("Moshi is replying...")
    silence_code = torch.zeros_like(all_codes[0]).to(DEVICE_HOME)
    
    eos_detected = False
    tail_frames = 0
    MAX_TAIL = 4  
    
    for _ in range(120): 
        tokens_out = lm_gen.step(silence_code)
        if tokens_out is not None: 
            text_token = tokens_out[0, 0].item()
            if text_token in (0, 3):
                eos_detected = True
            process_step(tokens_out)
            
            if eos_detected:
                tail_frames += 1
                if tail_frames >= MAX_TAIL:
                    break
    
final_moshi_codes = torch.cat(collected_moshi_codes, dim=-1) 

print("\n" + "="*40)
print("MOSHI'S TEXT RESPONSE:")
print("".join(generated_text_pieces).strip() or "(No text generated)")
print("="*40 + "\n")

# ==========================================
# 4. PHASE 2: BRIDGING & DSP POST-PROCESSING
# ==========================================
print("🌉 Phase 2: Applying True SoundStorm Bridge and DSP Filters...")

bridge = TrueSoundStormBridge().to(DEVICE_HOME)
try:
    raw_state_dict = torch.load(BRIDGE_PATH, map_location=DEVICE_HOME, weights_only=True)
    new_state_dict = {k.replace('module.', ''): v for k, v in raw_state_dict.items()}
    bridge.load_state_dict(new_state_dict)
except Exception as e:
    print(f"⚠️ Could not load bridge weights: {e}")
bridge.eval()

# Load Target Identity
sample_data = torch.load(TARGET_PT_PATH, weights_only=True)
id_vec = sample_data["identity_vector"].view(1, -1).to(DEVICE_HOME, dtype=torch.float32)

with torch.no_grad():
    cb0_sequence = final_moshi_codes[:, 0, :] 
    
    # 🛠️ True SoundStorm Inference with bfloat16 safety
    with autocast('cuda', dtype=torch.bfloat16):
        new_logits = bridge.forward_infer(cb0_sequence, id_vec) 
    
    # Argmax and transpose to match Moshi's spatial bounds: (batch, 7, seq_len)
    new_cb1_7 = torch.argmax(new_logits, dim=-1).transpose(1, 2) 
    
    hybrid_codes = final_moshi_codes.clone()
    hybrid_codes[:, 1:8, :] = new_cb1_7
    bridged_waveform = mimi.decode(hybrid_codes).to(torch.float32).cpu()

# DSP: Clean high-freq hiss and fade out tail
clean_waveform = torchaudio.functional.lowpass_biquad(bridged_waveform, sample_rate=24000, cutoff_freq=7500.0)

fade_samples = int(24000 * 0.2) 
if clean_waveform.shape[-1] > fade_samples:
    fade_curve = torch.linspace(1.0, 0.0, fade_samples) ** 2 
    clean_waveform[0, 0, -fade_samples:] *= fade_curve

torchaudio.save(OUTPUT_FILENAME, clean_waveform.squeeze(0), 24000)
print(f"🎉 Success! SoundStorm Bridged audio saved to: {OUTPUT_FILENAME}")
display(Audio(OUTPUT_FILENAME, rate=24000))

# ==========================================
# 5. PHASE 3: RESEARCH METRICS REPORT
# ==========================================
print("\n" + "="*50)
print("📊 FINAL RESEARCH METRICS REPORT")
print("="*50)

def get_prosody_stats(audio_path):
    y, sr = librosa.load(audio_path, sr=16000)
    f0, _, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
    f0 = f0[~np.isnan(f0)]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "mean_f0": np.mean(f0) if len(f0) > 0 else 0,
        "std_f0": np.std(f0) if len(f0) > 0 else 0,
        "mean_energy": np.mean(rms)
    }

target_stats = get_prosody_stats(REFERENCE_WAV)
bridge_stats = get_prosody_stats(OUTPUT_FILENAME)

print("1️⃣ PROSODY & EMPATHY ALIGNMENT")
print("-" * 45)
print(f"{'Metric':<18} | {'Target':<12} | {'Bridge Out':<12}")
print("-" * 45)
print(f"{'Mean Pitch (Hz)':<18} | {target_stats['mean_f0']:>12.2f} | {bridge_stats['mean_f0']:>12.2f}")
print(f"{'Pitch StdDev':<18} | {target_stats['std_f0']:>12.2f} | {bridge_stats['std_f0']:>12.2f}")
print(f"{'Mean Energy':<18} | {target_stats['mean_energy']:>12.6f} | {bridge_stats['mean_energy']:>12.6f}")
print("-" * 45)

try:
    from pymcd.mcd import Calculate_MCD
    mcd_toolbox = Calculate_MCD(MCD_mode="dtw")
    mcd_score = mcd_toolbox.calculate_mcd(REFERENCE_WAV, OUTPUT_FILENAME)
    print(f"\n2️⃣ ACOUSTIC DISTORTION")
    print("-" * 45)
    print(f"📉 Official DTW-MCD Score:      {mcd_score:.2f} dB")
except ImportError:
    print("📉 Please run `!pip install pymcd` to view MCD score.")

print(f"\n3️⃣ BIOMETRIC IDENTITY TRANSFER")
print("-" * 45)
try:
    # Initialize the VoiceEncoder
    encoder = VoiceEncoder()
    
    # Preprocess the audio files (Resemblyzer applies VAD and normalization)
    wav_target_processed = preprocess_wav(REFERENCE_WAV)
    wav_output_processed = preprocess_wav(OUTPUT_FILENAME)
    
    # Generate 256-dimensional d-vectors
    emb_target = encoder.embed_utterance(wav_target_processed)
    emb_generated = encoder.embed_utterance(wav_output_processed)
    
    # Calculate Cosine Similarity via inner product (vectors are L2 normalized)
    similarity = np.inner(emb_target, emb_generated)
    
    print(f"🧬 Identity Similarity Score: {similarity:.4f}")
    
    if similarity > 0.75:
        print("🟢 RESULT: Strong Identity Match!")
    elif similarity > 0.60:
        print("🟡 RESULT: Moderate Identity Match (Perceptually similar)")
    else:
        print("🔴 RESULT: Weak Match")
        
except Exception as e:
    print(f"⚠️ Identity Check Failed: {e}")
print(f"\n{'='*55}\n⚠️ RESEARCHER NOTE FOR EVALUATION: Due to Moshi's non-zero temperature sampling (temp=0.8), semantic generation is inherently stochastic. Each run produces slightly different word lengths and pitch contours. Therefore, these metrics will naturally fluctuate around the reported benchmarks.\n{'='*55}\n")

⚠️ Missing resemblyzer! Run: !pip install resemblyzer


ModuleNotFoundError: No module named 'moshi'